In [ ]:
import pandas as pd

SEASON = 2026
MIN_PA = 502

batting = pd.read_csv("../data/bsr_hits.csv")
guts = pd.read_csv("../data/fangraphs-guts-data.csv")
weights = guts.set_index("Season").loc[SEASON]


In [11]:
def score_players(data):
    if data["PlayerId"].duplicated().any():
        raise ValueError("Use one combined season row per player.")
    if data[["PA", "1B", "2B", "3B", "BsR"]].isna().any().any():
        raise ValueError("Fill in missing inputs from the source before scoring.")

    data = data.copy()
    data["hit_credit"] = (
        data["1B"] * weights["w1B"]
        + data["2B"] * weights["w2B"]
        + data["3B"] * weights["w3B"]
    ) / weights["wOBAScale"]
    data["simpson_credit"] = data["hit_credit"] + data["BsR"]
    pa = data["PA"].where(data["PA"] > 0)
    data["hit_credit_per_100_PA"] = 100 * data["hit_credit"] / pa
    data["BsR_per_100_PA"] = 100 * data["BsR"] / pa
    data["SIMPSON_v2"] = 100 * data["simpson_credit"] / pa
    return data


players = score_players(batting)
baseline = score_players(pd.read_csv(LEAGUE_FILE)) if LEAGUE_FILE else players
score = "SIMPSON_plus" if LEAGUE_FILE else "SIMPSON_sample_plus"

if baseline["PA"].sum() <= 0:
    raise ValueError("The baseline needs positive total PA.")
average = 100 * baseline["simpson_credit"].sum() / baseline["PA"].sum()
if average <= 1e-6:
    raise ValueError("The baseline rate is too small for a plus score.")

players[score] = 100 * players["SIMPSON_v2"] / average
print(f"{SEASON} {score} baseline: {average:.3f} per 100 PA")


2026 SIMPSON_sample_plus baseline: 15.386 per 100 PA


In [ ]:
leaderboard = players.loc[players["PA"] >= MIN_PA].sort_values(score, ascending=False)
columns = ["Name", "PA", "hit_credit_per_100_PA", "BsR_per_100_PA", "SIMPSON_v2", score]
display(leaderboard[columns].head(20).round(2))
display(players.loc[players["Name"].eq("Chandler Simpson"), columns].round(2))

,Name,PA,hit_credit_per_100_PA,BsR_per_100_PA,SIMPSON_v2,SIMPSON_sample_plus
5,Chandler Simpson,612,21.90,1.18,23.09,150.05
40,Luis Arraez,655,21.77,0.22,21.99,142.94
9,Otto Lopez,687,20.98,0.84,21.82,141.82
2,Jake McCarthy,596,20.13,1.43,21.56,140.14
58,Nick Gonzales,608,20.47,0.05,20.52,133.35
51,Jung Hoo Lee,579,19.26,0.12,19.38,125.96
62,Ceddanne Rafaela,576,19.19,0.03,19.23,124.95
0,Bobby Witt Jr.,632,17.47,1.47,18.94,123.08
75,Michael Harris II,621,18.97,-0.09,18.88,122.70
4,Bryson Stott,608,17.36,1.35,18.71,121.61


,Name,PA,hit_credit_per_100_PA,BsR_per_100_PA,SIMPSON_v2,SIMPSON_sample_plus
5,Chandler Simpson,612,21.9,1.18,23.09,150.05


In [14]:
leaderboard[columns].to_csv("../leaderboards/simpsonv2_leaderboard.csv", index=False)